<a href="https://colab.research.google.com/github/jiminmini/mini/blob/ESAA_OB/9_29_%EC%88%98%EC%83%81%EC%9E%91_%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**[주제 및 데이터]**
**주제**: Google Landmark Retrieval 2020 경진대회에서, 주어진 쿼리 이미지에 대해 동일한 랜드마크를 가지는 이미지들을 인덱스 데이터베이스에서 찾아내는 검색(retrieval) 문제.

**데이터 설명**:

데이터셋: Google Landmarks Dataset v2 (GLDv2) 기반

학습 이미지 수: cleaned GLDv2 기준으로 약 1,5M 이미지, 클래스 수 약 81,313개 등

인덱스 세트(index set)와 쿼리 이미지 세트가 별도로 제공

평가 지표: mAP@100 (mean average precision at top-100)

비랜드마크 이미지 (distractor) 존재 가능성 있음 (즉, 쿼리가 실제 랜드마크가 아닐 수도 있음)

논문에서는 후처리 없이도 좋은 성능 내는 전략 강조

---


#**[코드 리뷰]**
**1. EDA (탐색적 데이터 분석)**

각 랜드마크 클래스별 이미지 수 분포 확인 (클래스 불균형 점검)

이미지 해상도 분포, 이미지 품질 (흐림, 잘린 이미지 등) 점검

인덱스 세트 / 쿼리 세트 간의 클래스 공통성 여부 파악

distractor 이미지 비율 및 특성 파악

(논문에는 EDA 섹션 명시적 제목은 없지만, 전처리 및 학습 전략 설계 시 데이터 특성 인지가 바탕이 되었을 것으로 보임)

---


**2. 전처리**

이미지 크기 점진적 확대
 처음에는 낮은 해상도 (예: 448 × 448)으로 학습 시작 → 점차 512, 640, 736 등 더 높은 해상도로 fine-tuning 단계 진행함

증강 (Augmentation)
 좌우 반전 (left-right flip), RandomCrop, Brightness 조정, Color 변화, Cutout, Contrast, Shear, Rotate 등 다양한 augmentation을 단계별로 적용

ArcMargin Loss 사용
 클래스 분리를 강화하기 위해 softmax 대신 ArcMargin / angular margin 기반 손실 함수 사용

Margin 값 점진적 증가 전략
 학습이 진행될수록 margin 값을 키우는 방식 사용 (논문 강조)

모델 간소화 / 변환 처리
 PaddlePaddle / PyTorch 프레임워크로 학습 후 TensorFlow로 변환 등도 언급됨

---
**3. 모델링 (Modeling)**

백본 네트워크 (Backbone)
 EfficientNet 계열 등을 주요 백본으로 사용

헤드 구조 (Head / Embedding Layer)
 Global descriptor를 뽑기 위해 embedding 레이어 (예: 512차원) + 클래스 분류용 fc 레이어 병행

손실 함수
 ArcMargin Loss (margin 증가 전략 포함)

훈련 전략
 여러 스텝으로 fine-tune: 낮은 해상도로 시작 → 점진적으로 해상도 올리며 fine-tune

앙상블 (Ensemble)
 여러 모델 조합 사용. 서로 다른 백본 + 다른 해상도 + augmentation 조합 등을 섞어서 앙상블

후처리 없이 순수 Retrieval 방식
 이 솔루션은 논문 제목대로 “without post-processing” 전략을 강조함 — 즉, spatial verification 등 복잡한 로컬 피처 매칭 트릭 없이 성능을 끌어올리는 데 집중함

---
#**[차별 점 및 배울 점]**

✅ 차별 점

후처리 (spatial verification, 로컬 피처 매칭 등) 없이도 좋은 성능을 내는 retrieval 중심 접근

Margin 값을 점진적으로 증가시키는 학습 전략

이미지 해상도를 점진적으로 키우는 multi-stage fine-tuning 전략

다양한 augmentation & 모델 조합을 통한 앙상블 전략

✨ 배울 점

단순히 복잡한 후처리를 많이 쓰기보다는, 전처리 + 학습 전략 + 손실 함수 설계 만으로도 강한 모델을 만드는 방법

해상도 변화와 margin 조절 같은 학습 스케줄링 전략이 성능에 미치는 영향

앙상블 다양성의 중요성 (백본 다양성, 해상도 다양성 등)

retrieval 문제 특유의 특성 (distractor suppression, 클래스 분리 등)에 맞춘 손실 함수 / 학습 방식 설계

